# grad-accumulate-on-leaf — ex1: accumulate_grad: leaf.grad = (leaf.grad or 0) + g

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-accumulate-on-leaf`. Running the final beacon cell reports progress against the `Backprop: Grad accumulate on leaf` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Grad accumulate on leaf — quick refresher

When the reverse pass reaches a **leaf** (`.recipe is None`, `.requires_grad is True`), it must **accumulate** the incoming gradient into `leaf.grad`. Two cases:

- **First time we touch this leaf** (`leaf.grad is None`): set   `leaf.grad = g`.
- **Second+ time** (some other path through the graph already   reached it): `leaf.grad = leaf.grad + g`.

Canonical form:
```python
def accumulate_grad(leaf: Tensor, g: Tensor) -> None:
    if leaf.grad is None:
        leaf.grad = g
    else:
        leaf.grad = leaf.grad + g
```

Why ACCUMULATE and not overwrite: a single Tensor can appear multiple times in the graph (e.g. `y = w * w` — `w` is a parent of `y` twice). Each path through the graph contributes its own `dL/dw`; the *total* derivative is the sum. Overwriting would keep only the last-visited path's contribution.

This is why PyTorch's `.backward()` accumulates into `.grad` and why you must `optimizer.zero_grad()` between training steps.

### Exercise 1 — accumulate_grad: leaf.grad = (leaf.grad or 0) + g

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the leaf-grad accumulation pattern: set leaf.grad on first visit, add to it on subsequent visits — the standard branching pattern that requires .zero_grad() between training steps.
> Keywords: accumulate, grad, leaf, first-touch, zero-grad
> ```

**KCs targeted:** `grad-accumulate-on-leaf`, `parameter-subclass-of-tensor`

Implement `accumulate_grad(leaf: MiniTensor, g: torch.Tensor) -> None`. The reverse pass calls this every time it reaches a leaf (`leaf.requires_grad is True`, `leaf.recipe is None`). Two cases:

**1. First time we touch this leaf** (`leaf.grad is None`):
   Set `leaf.grad = g`.

**2. Second+ time we touch it** (`leaf.grad is not None`):
   Set `leaf.grad = leaf.grad + g`.

Notes:

- **Mutates `leaf.grad`; returns `None`.** (Caller doesn't need the value.)
- **Use `+`, not `+=`.** Re-bind `leaf.grad` to a new tensor rather than mutating the existing grad tensor in place. Some tests check that an externally-held reference to the old grad tensor is NOT mutated — important for the `optimizer.step()` case where the grad tensor is read elsewhere.
- **Shape: `g.shape == leaf.array.shape`** (caller's responsibility — earlier `unbroadcast` step handles this). Trust it.

**Why accumulate.** A single Tensor can appear multiple times in the compute graph — e.g. `y = w * w` makes `w` a parent of `y` twice (argnum 0 AND argnum 1). The total derivative `dL/dw = dL/dy * 2w` is the SUM of contributions from each path. Overwriting would keep only the last-visited path's contribution.

This is why PyTorch's `.backward()` accumulates into `.grad` and why training loops must call `optimizer.zero_grad()` (or manually set `p.grad = None`) between steps — otherwise gradients from previous steps stay around and corrupt the next update.

In [ ]:
def accumulate_grad(leaf: MiniTensor, g) -> None:
    """Set leaf.grad = g if None else leaf.grad + g. Mutates leaf; no return."""
    raise NotImplementedError()


def _test_ex1():
    # --- first-touch: leaf.grad is None → set to g ---
    leaf = MiniTensor(t.zeros(3), requires_grad=True)
    assert leaf.grad is None, 'precondition'
    g = t.tensor([1.0, 2.0, 3.0])
    ret = accumulate_grad(leaf, g)

    assert ret is None, 'accumulate_grad must return None (mutates in place)'
    assert leaf.grad is not None
    assert t.allclose(leaf.grad, t.tensor([1.0, 2.0, 3.0])), (
        f'first-touch must set leaf.grad = g, got {leaf.grad}'
    )

    # --- second-touch: leaf.grad is not None → ADD g ---
    g2 = t.tensor([10.0, 20.0, 30.0])
    accumulate_grad(leaf, g2)
    assert t.allclose(leaf.grad, t.tensor([11.0, 22.0, 33.0])), (
        f'second-touch must ADD g, got {leaf.grad}'
    )

    # --- many touches: sum-like accumulation across an iteration ---
    leaf2 = MiniTensor(t.zeros(4), requires_grad=True)
    for i in range(5):
        accumulate_grad(leaf2, t.ones(4))
    assert t.allclose(leaf2.grad, t.full((4,), 5.0)), (
        f'5x ones should accumulate to 5: got {leaf2.grad}'
    )

    # --- rebind semantics: external reference to OLD grad is not mutated ---
    # This is the critical safety property — the optimizer's reference to
    # leaf.grad must NOT silently change underneath it.
    leaf3 = MiniTensor(t.zeros(3), requires_grad=True)
    accumulate_grad(leaf3, t.tensor([1.0, 2.0, 3.0]))
    old_ref = leaf3.grad
    old_ref_clone = old_ref.clone()
    accumulate_grad(leaf3, t.tensor([10.0, 20.0, 30.0]))
    assert t.allclose(old_ref, old_ref_clone), (
        'externally-held reference to leaf.grad must NOT be mutated in place — '
        "use `leaf.grad = leaf.grad + g`, NOT `leaf.grad += g`"
    )
    assert leaf3.grad is not old_ref, (
        'leaf.grad must REBIND to a new tensor, not mutate the old one'
    )

    # --- the canonical `y = w * w` use case ---
    # w appears at argnum 0 AND argnum 1 of multiply → reverse visits w twice.
    # total dL/dw = dL/dy * 2w; accumulating from both paths gives the right total.
    w = MiniTensor(t.tensor([3.0]), requires_grad=True)
    # Path 1: contribution from arg-0 of multiply (= grad_out * w = 1 * 3 = 3)
    accumulate_grad(w, t.tensor([3.0]))
    # Path 2: contribution from arg-1 of multiply (= grad_out * w = 1 * 3 = 3)
    accumulate_grad(w, t.tensor([3.0]))
    # Total should be 6 — same as torch.autograd would give for d(w^2)/dw = 2w = 6
    assert t.allclose(w.grad, t.tensor([6.0])), (
        f'y = w * w → dL/dw should be 2w = 6 via 2-path accumulation, got {w.grad}'
    )
    # Cross-check against torch.autograd for confidence.
    w_ref = t.tensor([3.0], requires_grad=True)
    y = w_ref * w_ref
    y.sum().backward()
    assert t.allclose(w.grad, w_ref.grad), (
        f'accumulation must match torch.autograd: ours={w.grad}, ref={w_ref.grad}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def accumulate_grad(leaf: MiniTensor, g) -> None:
    if leaf.grad is None:
        # First-touch: set directly (no allocation for an initial zero).
        leaf.grad = g
    else:
        # Subsequent touches: REBIND (NOT in-place +=).
        # Rebinding leaves any externally-held reference to the old grad
        # untouched — important if the optimizer is holding the grad.
        leaf.grad = leaf.grad + g
```

**Why `+`, not `+=`.** `leaf.grad += g` mutates the existing grad tensor in place. If anything else holds a reference to that tensor (e.g. `optimizer.step()` snapshotted `p.grad` before the accumulate ran), the snapshot silently changes underneath. Using `+` rebinds `leaf.grad` to a fresh tensor, leaving the old one alone.

**Why first-touch sets, not adds-to-zeros.** Skips an unnecessary `t.zeros_like(g)` allocation on every leaf's first visit. Across a model with thousands of parameters, the allocation cost matters.

**Why `zero_grad()` exists.** Because `accumulate_grad` ALWAYS adds (never overwrites), the previous step's gradient stays in `leaf.grad` forever unless explicitly cleared. This is why PyTorch's training loop has the canonical `optimizer.zero_grad()` line — it sets every `p.grad = None` (or zeros it), so the next backward starts from a clean state.

**The shared-parent case is the whole reason for this design.** `y = w * w` has `w` as parent twice. `y = a * b` then `z = y + b` has `b` as parent of two different intermediates. Real models share parameters across layers (e.g. weight-tied embedding and unembedding). Each path contributes a separate gradient; the total derivative is the sum.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()